In [ ]:
!pip install be-great
!pip install peft

In [ ]:
from be_great import GReaT
import numpy as np
import pandas as pd
import torch

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

df_train = pd.read_csv("/content/train.csv")
df_test  = pd.read_csv("/content/test.csv")

cond_cols = ["Frequency (Hz)", "Storage modulus (Pa)", "Loss modulus (Pa)"]
other_cols = [c for c in df_train.columns if c not in cond_cols]

ordered_cols = cond_cols + other_cols

df_train_ord = df_train[ordered_cols].copy()
df_test_ord  = df_test[ordered_cols].copy()

temp = df_test_ord.copy()
temp[other_cols] = np.nan

model = GReaT(
    llm="tabularisai/Qwen3-0.3B-distil",
    float_precision=2,
    batch_size=8,
    epochs=10,
    efficient_finetuning="lora",
    fp16=False
)
model.fit(df_train_ord)

synth = model.impute(temp, max_length=256)

synth[cond_cols] = df_test_ord[cond_cols].values

synth.to_csv("/content/synth_inverse_design.csv", index=False)

Loading weights:   0%|          | 0/156 [00:00<?, ?it/s]

trainable params: 1,146,880 || all params: 376,963,584 || trainable%: 0.3042


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
500,1.282163
1000,0.898424
1500,0.893561
2000,0.880124
2500,0.867133
3000,0.866503
3500,0.860239
4000,0.862337
4500,0.850005


100%|██████████| 234/234 [27:18<00:00,  7.00s/it]
